### Functions & define

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    # Load session
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    # spike data (unit, trial, time)
    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    # behavioral data
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    # Unit Info
    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    # Time Info
    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')

    # Prepare data
    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes
    }
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'dual': dual_units
    }
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_info': time_info,
        'time_info_tuning': time_info_tuning
    }


def calculate_decision_variable(y_test, y_proba, epsilon=1e-10):
    """
    Calculate decision variable from classifier probabilities
    
    Args:
        y_test: True label (0 or 1)
        y_proba: Classifier probability for class 1
        epsilon: Small value to avoid log(0)
    
    Returns:
        dv: Decision variable (positive = confident, negative = uncertain)
    """
    # Ensure y_proba is in valid range [epsilon, 1-epsilon]
    y_proba_clipped = np.clip(y_proba, epsilon, 1 - epsilon)
    
    # Calculate log-odds (decision variable)
    dv = np.log(y_proba_clipped / (1 - y_proba_clipped))
    
    # If true class is 0, flip the sign
    if y_test == 0:
        dv = -dv
    
    return dv

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

subject = "zarya"
date = "20250602"
data_dict = load_session_data(subject, date)
time_axes_dots3DMP = data_dict['time_axes_dots3DMP']

# Define brain areas, alignments, and targets
areas = ['dual', 'MST', 'VPS']
alignments = ['stimOn', 'saccOnset', 'postTargHold']
targets = ['choice', 'PDW', 'stimulus']
area_colors = {'dual': 'blue', 'MST': 'red', 'VPS': 'black'}

# Alignment display names
alignment_names = {
    'stimOn': 'Stimulus On',
    'saccOnset': 'Choice', 
    'postTargHold': 'Confidence'
}
# Define modality/coherence conditions with their file parameters
modality_conditions = {
    'Vestibular Only': {'mod': 1, 'coh': 1},        # Row 0: mod1_coh1
    'Visual Low Coh': {'mod': 2, 'coh': 1},         # Row 1: mod2_coh1
    'Visual High Coh': {'mod': 2, 'coh': 2},        # Row 2: mod2_coh2
    'Combined Low Coh': {'mod': 3, 'coh': 1},       # Row 3: mod3_coh1
    'Combined High Coh': {'mod': 3, 'coh': 2}       # Row 4: mod3_coh2       
}

# Velocity profile markers for stimOn alignment
vel_markers = {
    'max_velocity': 0.66,
    'max_acceleration': 0.4,
}
# Define heading groups
heading_groups = {
    '±10°': [1, 7],    # Most extreme headings
    '±3.9°': [2, 6],     # Medium-high headings  
    '±1.5°': [3, 5],   # Medium-low headings
    '0°': [4],         # Zero heading (ambiguous)
    'All': [1, 2, 3, 4, 5, 6, 7]  # All headings combined
}
# Define session dates to analyze
dates = ["20250602", "20250523","20250702", "20250710"]   

# Defualt function to create file path
def get_filepath(date, area, alignment, target):
    return f"D:\\Neural-Pipeline\\results\\analysis_population\\decoders\\zarya_{date}_{area}_{target}_{alignment}_train_mod3_coh2_test_mod3_coh2_cv_results.npy"


### plot results by modality and coherence, Prob and AUC

In [ ]:



# Load session data once outside the loop (since it's the same across sessions)
subject = "zarya"
date = "20250602"  # Reference date for loading behavioral structure
data_dict = load_session_data(subject, date)

# Define brain areas, alignments, and targets
areas = ['dual', 'MST', 'VPS']
alignments = ['stimOn', 'saccOnset', 'postTargHold']
targets = ['choice', 'stimulus', 'PDW']
area_colors = {'dual': 'blue', 'MST': 'red', 'VPS': 'black'}

# Alignment display names
alignment_names = {
    'stimOn': 'Stimulus On',
    'saccOnset': 'Choice', 
    'postTargHold': 'Confidence'
}

# Define modality/coherence conditions with their file parameters
modality_conditions = {
    'Vestibular Only': {'mod': 1, 'coh': 1},        # Row 0: mod1_coh1
    'Visual Low Coh': {'mod': 2, 'coh': 1},         # Row 1: mod2_coh1
    'Visual High Coh': {'mod': 2, 'coh': 2},        # Row 2: mod2_coh2
    'Combined Low Coh': {'mod': 3, 'coh': 1},       # Row 3: mod3_coh1
    'Combined High Coh': {'mod': 3, 'coh': 2}       # Row 4: mod3_coh2
}

# Velocity profile markers for stimOn alignment
vel_markers = {
    'max_velocity': 0.66,
    'max_acceleration': 0.4,
}
dates = ["20250602", "20250702", "20250710", "20250523"]   

# Function to create file path based on modality/coherence condition
def get_filepath(date, area, alignment, target, mod, coh):
    return f"D:\\Neural-Pipeline\\results\\analysis_population\\decoders\\zarya_{date}_{area}_{target}_{alignment}_train_mod{mod}_coh{coh}_test_mod{mod}_coh{coh}_cv_results.npy"

# Process each target separately
for target in targets:
    print(f"\n{'#'*60}")
    print(f"PROCESSING TARGET: {target.upper()} - BY MODALITY/COHERENCE")
    print(f"{'#'*60}")
    
    # Create figure with subplots (5 rows x 6 columns: 5 conditions x (3 prob + 3 AUC))
    fig, axes = plt.subplots(5, 6, figsize=(30, 25))
    
    all_stats = {}
    
    for j, alignment in enumerate(alignments):
        print(f"\n{'='*20} {alignment.upper()} {'='*20}")
        
        alignment_stats = {}
        
        # Initialize time_axes
        time_axes = None
        
        for area in areas:
            color = area_colors[area]
            
            print(f"\n{area}:")
            
            # Process each modality/coherence condition (each row)
            for i, (condition_name, condition_params) in enumerate(modality_conditions.items()):
                mod = condition_params['mod']
                coh = condition_params['coh']
                
                print(f"\n  {condition_name} (mod={mod}, coh={coh}):")
                
                # Collect trials from all dates for this specific condition
                all_trials = []
                successful_dates = 0
                
                for date in dates:
                    # Get time axes for this alignment (from loaded session data)
                    try:
                        if date == "20250602":  # Use pre-loaded data for reference date
                            time_axes_dots3DMP = data_dict['time_axes_dots3DMP']
                        else:
                            # Load data for other dates
                            date_data_dict = load_session_data(subject, date)
                            time_axes_dots3DMP = date_data_dict['time_axes_dots3DMP']
                        
                        if time_axes is None:
                            time_axes = time_axes_dots3DMP[alignment]
                        
                        print(f"    {date}: Loaded session data successfully")
                    except Exception as e:
                        print(f"    {date}: Failed to load session data - {str(e)}")
                        continue
                    
                    # Get the correct filepath for this condition
                    filepath = get_filepath(date, area, alignment, target, mod, coh)
                    
                    try:
                        # Load results
                        results = np.load(filepath, allow_pickle=True).item()
                        trial_results = results['trial_results']
                        
                        # Add all trials to the pool
                        all_trials.extend(trial_results)
                        
                        successful_dates += 1
                        print(f"    {date}: loaded {len(trial_results)} trials")
                        
                    except FileNotFoundError:
                        print(f"    {date}: File not found - {filepath}")
                        continue
                    except Exception as e:
                        print(f"    {date}: Error - {str(e)}")
                        continue
                
                if successful_dates == 0:
                    print(f"    No data found for {area} {condition_name}")
                    continue
                
                print(f"    Total trials: {len(all_trials)} from {successful_dates} dates")
                
                # Set target-specific labels
                if target == 'choice':
                    class_0_label = f'{area} Left Choice'
                    class_1_label = f'{area} Right Choice'
                    ylabel = 'Probability of Right Choice'
                elif target == 'stimulus':
                    class_0_label = f'{area} Left Heading'
                    class_1_label = f'{area} Right Heading'
                    ylabel = 'Probability of Right Heading'
                elif target == 'PDW':
                    class_0_label = f'{area} Low Confidence'
                    class_1_label = f'{area} High Confidence'
                    ylabel = 'Probability of High Confidence'
                
                # Process trials for this condition
                if all_trials:
                    time_points = sorted(list(set([trial['time'] for trial in all_trials])))
                    
                    class_0_proba = []
                    class_0_std = []
                    class_1_proba = []
                    class_1_std = []
                    pooled_mean_auc = []
                    pooled_std_auc = []
                    
                    for t in time_points:
                        time_trials = [trial for trial in all_trials if trial['time'] == t]
                        
                        if time_trials:
                            class_0_probabilities = []
                            class_1_probabilities = []
                            
                            for trial in time_trials:
                                # Handle both old and new formats
                                if isinstance(trial['y_test'], (list, np.ndarray)) and len(trial['y_test']) > 1:
                                    # Old format: multiple values per trial
                                    y_test = trial['y_test']
                                    y_proba = trial['y_proba']
                                    
                                    for k, actual_class in enumerate(y_test):
                                        if actual_class == 0:
                                            class_0_probabilities.append(y_proba[k])
                                        elif actual_class == 1:
                                            class_1_probabilities.append(y_proba[k])
                                else:
                                    # New format: single value per trial
                                    y_test = trial['y_test'][0] if isinstance(trial['y_test'], (list, np.ndarray)) else trial['y_test']
                                    y_proba = trial['y_proba'][0] if isinstance(trial['y_proba'], (list, np.ndarray)) else trial['y_proba']
                                    
                                    if y_test == 0:
                                        class_0_probabilities.append(y_proba)
                                    elif y_test == 1:
                                        class_1_probabilities.append(y_proba)
                            
                            # Calculate statistics
                            if class_0_probabilities:
                                class_0_proba.append(np.mean(class_0_probabilities))
                                class_0_std.append(np.std(class_0_probabilities) / np.sqrt(len(class_0_probabilities)))
                            else:
                                class_0_proba.append(np.nan)
                                class_0_std.append(0)
                            
                            if class_1_probabilities:
                                class_1_proba.append(np.mean(class_1_probabilities))
                                class_1_std.append(np.std(class_1_probabilities) / np.sqrt(len(class_1_probabilities)))
                            else:
                                class_1_proba.append(np.nan)
                                class_1_std.append(0)
                            
                            # AUC calculation
                            time_aucs = [trial['auc'] for trial in time_trials]
                            pooled_mean_auc.append(np.mean(time_aucs))
                            pooled_std_auc.append(np.std(time_aucs) / np.sqrt(len(time_aucs)))
                        else:
                            class_0_proba.append(np.nan)
                            class_0_std.append(0)
                            class_1_proba.append(np.nan)
                            class_1_std.append(0)
                            pooled_mean_auc.append(0.5)
                            pooled_std_auc.append(0)
                    
                    # Match lengths with proper time axes
                    if time_axes is not None and len(time_axes) != len(class_0_proba):
                        if len(time_axes) > len(class_0_proba):
                            time_axes_condition = time_axes[:len(class_0_proba)]
                        else:
                            time_axes_condition = np.linspace(time_axes[0], time_axes[-1], len(class_0_proba))
                    elif time_axes is not None:
                        time_axes_condition = time_axes
                    else:
                        time_axes_condition = np.arange(len(class_0_proba))
                        print(f"      Warning: Using default time axes for {area}")
                    
                    # Plot Probability (left panel: columns 0, 1, 2)
                    ax_prob = axes[i, j]
                    
                    # Plot class 0 probabilities
                    ax_prob.plot(time_axes_condition, class_0_proba, color=color, linestyle='--', 
                               linewidth=2, label=class_0_label, alpha=0.8)
                    ax_prob.fill_between(time_axes_condition, 
                                       np.array(class_0_proba) - np.array(class_0_std),
                                       np.array(class_0_proba) + np.array(class_0_std),
                                       alpha=0.05, color=color)
                    
                    # Plot class 1 probabilities
                    ax_prob.plot(time_axes_condition, class_1_proba, color=color, linestyle='-', 
                               linewidth=3, label=class_1_label, alpha=0.8)
                    ax_prob.fill_between(time_axes_condition, 
                                       np.array(class_1_proba) - np.array(class_1_std),
                                       np.array(class_1_proba) + np.array(class_1_std),
                                       alpha=0.05, color=color)
                    
                    # Plot AUC (right panel: columns 3, 4, 5)
                    ax_auc = axes[i, j + 3]
                    ax_auc.plot(time_axes_condition, pooled_mean_auc, color=color, linestyle='-', 
                              linewidth=3, label=f'{area}', alpha=0.8)
                    ax_auc.fill_between(time_axes_condition, 
                                       np.array(pooled_mean_auc) - np.array(pooled_std_auc),
                                       np.array(pooled_mean_auc) + np.array(pooled_std_auc),
                                       alpha=0.1, color=color)
        
        # Format all subplots for this alignment (column j for prob, j+3 for AUC)
        for i, condition_name in enumerate(modality_conditions.keys()):
            # Format Probability subplot
            ax_prob = axes[i, j]
            ax_prob.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
            ax_prob.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            
            # Add velocity markers for stimOn alignment with text annotations
            if alignment == 'stimOn':
                ax_prob.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
                ax_prob.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
                
                # Add text annotations at the top of the plot
                ax_prob.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='orange', fontsize=10, transform=ax_prob.get_xaxis_transform())
                ax_prob.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='purple', fontsize=10, transform=ax_prob.get_xaxis_transform())
            
            # Add text annotation for alignment line with proper alignment name
            ax_prob.text(0, 0.95, alignment_names[alignment], rotation=90, verticalalignment='top', 
                       horizontalalignment='right', color='gray', fontsize=10,
                       transform=ax_prob.get_xaxis_transform())
            
            ax_prob.set_xlabel('Time (s)')
            ax_prob.set_ylabel(ylabel)
            
            # Add titles
            if j == 0:  # First column only
                ax_prob.set_title(f'{condition_name}', fontsize=12, loc='left')
            if i == 0:  # First row only
                ax_prob.set_title(f'{alignment_names[alignment]}', fontsize=14)
            
            # Only show legend on the last probability subplot (bottom right of prob panels)
            if i == len(modality_conditions) - 1 and j == len(alignments) - 1:
                ax_prob.legend(fontsize=8)
            
            ax_prob.set_ylim(0.0, 1.0)
            
            # Format AUC subplot
            ax_auc = axes[i, j + 3]
            ax_auc.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
            ax_auc.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            
            # Add velocity markers for stimOn alignment with text annotations
            if alignment == 'stimOn':
                ax_auc.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
                ax_auc.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
                
                # Add text annotations at the top of the plot
                ax_auc.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='orange', fontsize=10, transform=ax_auc.get_xaxis_transform())
                ax_auc.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='purple', fontsize=10, transform=ax_auc.get_xaxis_transform())
            
            # Add text annotation for alignment line with proper alignment name
            ax_auc.text(0, 0.95, alignment_names[alignment], rotation=90, verticalalignment='top', 
                       horizontalalignment='right', color='gray', fontsize=10,
                       transform=ax_auc.get_xaxis_transform())
            
            ax_auc.set_xlabel('Time (s)')
            ax_auc.set_ylabel('AUC')
            
            # Add titles for AUC panels
            if i == 0:  # First row only
                ax_auc.set_title(f'{alignment_names[alignment]} - AUC', fontsize=14)
            
            # Only show legend on the last AUC subplot (bottom right of AUC panels)
            if i == len(modality_conditions) - 1 and j == len(alignments) - 1:
                ax_auc.legend(fontsize=8)
            
            ax_auc.set_ylim(0.4, 1.0)
    
    # Add overall title
    fig.suptitle(f'{target.upper()} Decoding by Modality/Coherence\nLeft: Probabilities, Right: AUC', 
                 fontsize=16, y=0.98)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.95, left=0.05)
    
    # Save the figure
    save_filename = f"D:\\Neural-Pipeline\\results\\analysis_population\\compare_area_{target}_zarya_by_modality_coherence.png"
    plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    print(f"\nFigure saved as: {save_filename}")
    
    plt.show()

### plot results by heading, prob and AUC

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

# Function to create file path
def get_filepath(date, area, alignment, target):
    return f"D:\\Neural-Pipeline\\results\\analysis_population\\decoders\\zarya_{date}_{area}_{target}_{alignment}_train_mod3_coh2_test_mod3_coh2_cv_results.npy"

# Process each target separately
for target in targets:
    print(f"\n{'#'*60}")
    print(f"PROCESSING TARGET: {target.upper()} - BY HEADING DIFFICULTY")
    print(f"{'#'*60}")
    
    # Skip 0° heading for stimulus target
    current_heading_groups = heading_groups.copy()
    if target == 'stimulus':
        current_heading_groups = {k: v for k, v in heading_groups.items() if k != '0°'}
        print("Skipping 0° heading for stimulus target")
    
    # Create figure with subplots (rows x 6 columns: heading groups x (3 prob + 3 AUC))
    n_rows = len(current_heading_groups)
    fig, axes = plt.subplots(n_rows, 6, figsize=(30, 5*n_rows))
    
    # Handle single row case
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    all_stats = {}
    
    for j, alignment in enumerate(alignments):
        print(f"\n{'='*20} {alignment.upper()} {'='*20}")
        
        alignment_stats = {}
        
        # Initialize time_axes
        time_axes = None
        
        for area in areas:
            color = area_colors[area]
            
            # Collect trials for all heading groups
            all_trials_by_group = {}
            for group_name, heading_indices in current_heading_groups.items():
                all_trials_by_group[group_name] = []
            
            print(f"\n{area}:")
            successful_dates = 0
            
            for date in dates:
                # Load session data for this specific date
                try:

                    time_axes_dots3DMP = data_dict['time_axes_dots3DMP']
                    
                    # Get time axes for this alignment (from loaded session data)
                    if time_axes is None:
                        time_axes = time_axes_dots3DMP[alignment]
                    
                    print(f"  {date}: Loaded session data successfully")
                except Exception as e:
                    print(f"  {date}: Failed to load session data - {str(e)}")
                    continue
                
                filepath = get_filepath(date, area, alignment, target)
                
                try:
                    # Load decoder results
                    results = np.load(filepath, allow_pickle=True).item()
                    trial_results = results['trial_results']
                    
                    # Filter trials by heading difficulty using the stored behavioral data
                    for trial in trial_results:
                        # Check if trial has the new format with behavioral data
                        if 'test_behavior' in trial:
                            # NEW FORMAT: Extract heading info directly from trial results
                            headings = trial['test_behavior']['headingInd']
                            
                            # Create separate trial entries for each test trial
                            for i, heading in enumerate(headings):
                                # Create a single-trial entry
                                single_trial = {
                                    'time': trial['time'],
                                    'cv_fold': trial['cv_fold'],
                                    'y_proba': trial['y_proba'][i:i+1],  # Single prediction
                                    'y_pred': trial['y_pred'][i:i+1],
                                    'y_test': trial['y_test'][i:i+1],
                                    'auc': trial['auc'],  # Keep overall AUC (but won't use for filtered conditions)
                                    'accuracy': trial['accuracy'],  # Keep overall accuracy
                                    # Store all behavioral info for this specific trial
                                    'heading': heading,
                                    'choice': trial['test_behavior']['choice'][i],
                                    'PDW': trial['test_behavior']['PDW'][i],
                                    'modality': trial['test_behavior']['modality'][i],
                                    'coherenceInd': trial['test_behavior']['coherenceInd'][i],
                                    'correct': trial['test_behavior']['correct'][i],
                                    'oneTargConf': trial['test_behavior']['oneTargConf'][i],
                                    'RT': trial['test_behavior']['RT'][i],
                                }
                                
                                # Add trial to appropriate groups
                                for group_name, heading_indices in current_heading_groups.items():
                                    if heading in heading_indices:
                                        all_trials_by_group[group_name].append(single_trial)
                        
                        else:
                            # OLD FORMAT: Skip or handle differently
                            print(f"  Warning: Old format detected for {date}, skipping...")
                            continue
                    
                    successful_dates += 1
                    
                    # Print trial counts for each group
                    for group_name in current_heading_groups.keys():
                        group_count = len([t for t in trial_results if 'test_behavior' in t 
                                         for h in t['test_behavior']['headingInd'] 
                                         if h in current_heading_groups[group_name]])
                        print(f"  {date}: {group_name} trials: {group_count}")
                    
                except FileNotFoundError:
                    print(f"  {date}: File not found - {filepath}")
                    continue
                except Exception as e:
                    print(f"  {date}: Error - {str(e)}")
                    continue
            
            if successful_dates == 0:
                print(f"  No data found for {area}")
                continue
            
            # Print total trial counts
            for group_name, trials in all_trials_by_group.items():
                print(f"  Total {group_name} trials: {len(trials)}")
            
            # Set ylabel once per area (will be used for both panels)
            if target == 'choice':
                ylabel = 'Probability of Right Choice'
            elif target == 'stimulus':
                ylabel = 'Probability of Right Heading'
            elif target == 'PDW':
                ylabel = 'Probability of High Confidence'
            
            # Define function to process trials and plot probabilities
            def process_and_plot_probabilities(ax, trials, group_name):
                if trials:
                    time_points = sorted(list(set([trial['time'] for trial in trials])))
                    
                    class_0_proba = []
                    class_0_std = []
                    class_1_proba = []
                    class_1_std = []
                    
                    for t in time_points:
                        time_trials = [trial for trial in trials if trial['time'] == t]
                        
                        if time_trials:
                            class_0_probabilities = []
                            class_1_probabilities = []
                            
                            for trial in time_trials:
                                y_test = trial['y_test'][0]  # Single value now
                                y_proba = trial['y_proba'][0]  # Single value now
                                
                                if y_test == 0:
                                    class_0_probabilities.append(y_proba)
                                elif y_test == 1:
                                    class_1_probabilities.append(y_proba)
                            
                            # Calculate statistics
                            if class_0_probabilities:
                                class_0_proba.append(np.mean(class_0_probabilities))
                                class_0_std.append(np.std(class_0_probabilities) / np.sqrt(len(class_0_probabilities)))
                            else:
                                class_0_proba.append(np.nan)
                                class_0_std.append(0)
                            
                            if class_1_probabilities:
                                class_1_proba.append(np.mean(class_1_probabilities))
                                class_1_std.append(np.std(class_1_probabilities) / np.sqrt(len(class_1_probabilities)))
                            else:
                                class_1_proba.append(np.nan)
                                class_1_std.append(0)
                        else:
                            class_0_proba.append(np.nan)
                            class_0_std.append(0)
                            class_1_proba.append(np.nan)
                            class_1_std.append(0)
                    
                    # Match lengths with proper time axes
                    if time_axes is not None and len(time_axes) != len(class_0_proba):
                        if len(time_axes) > len(class_0_proba):
                            time_axes_group = time_axes[:len(class_0_proba)]
                        else:
                            time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(class_0_proba))
                    elif time_axes is not None:
                        time_axes_group = time_axes
                    else:
                        time_axes_group = np.arange(len(class_0_proba))
                    
                    # Set target-specific labels
                    if target == 'choice':
                        class_0_label = f'{area} Left Choice'
                        class_1_label = f'{area} Right Choice'
                    elif target == 'stimulus':
                        class_0_label = f'{area} Left Heading'
                        class_1_label = f'{area} Right Heading'
                    elif target == 'PDW':
                        class_0_label = f'{area} Low Confidence'
                        class_1_label = f'{area} High Confidence'
                    
                    # Plot class 0 probabilities
                    ax.plot(time_axes_group, class_0_proba, color=color, linestyle='--', 
                               linewidth=2, label=class_0_label, alpha=0.8)
                    ax.fill_between(time_axes_group, 
                                       np.array(class_0_proba) - np.array(class_0_std),
                                       np.array(class_0_proba) + np.array(class_0_std),
                                       alpha=0.05, color=color)
                    
                    # Plot class 1 probabilities
                    ax.plot(time_axes_group, class_1_proba, color=color, linestyle='-', 
                               linewidth=3, label=class_1_label, alpha=0.8)
                    ax.fill_between(time_axes_group, 
                                       np.array(class_1_proba) - np.array(class_1_std),
                                       np.array(class_1_proba) + np.array(class_1_std),
                                       alpha=0.05, color=color)
                    
                    return time_axes_group
                return None

            # Define function to process trials and plot AUC - CORRECTED VERSION
            def process_and_plot_auc(ax, trials, group_name):
                if trials:
                    time_points = sorted(list(set([trial['time'] for trial in trials])))
                    
                    auc_values = []
                    auc_std = []
                    
                    for t in time_points:
                        time_trials = [trial for trial in trials if trial['time'] == t]
                        
                        if len(time_trials) >= 2:  # Need at least 2 trials
                            # Collect y_true and y_proba for this time point and heading group
                            y_true = []
                            y_proba = []
                            
                            for trial in time_trials:
                                y_true.append(trial['y_test'][0])
                                y_proba.append(trial['y_proba'][0])
                            
                            # Convert to numpy arrays
                            y_true = np.array(y_true)
                            y_proba = np.array(y_proba)
                            
                            # Check if we have both classes
                            unique_classes = np.unique(y_true)
                            if len(unique_classes) > 1:
                                try:
                                    # Calculate AUC for this specific heading group and time point
                                    overall_auc = roc_auc_score(y_true, y_proba)
                                    auc_values.append(overall_auc)
                                    
                                    # Calculate std across CV folds if available
                                    cv_folds = [trial['cv_fold'] for trial in time_trials]
                                    unique_folds = list(set(cv_folds))
                                    
                                    fold_aucs = []
                                    for fold in unique_folds:
                                        fold_trials = [trial for trial in time_trials if trial['cv_fold'] == fold]
                                        if len(fold_trials) >= 2:
                                            fold_y_true = np.array([trial['y_test'][0] for trial in fold_trials])
                                            fold_y_proba = np.array([trial['y_proba'][0] for trial in fold_trials])
                                            
                                            if len(np.unique(fold_y_true)) > 1:
                                                try:
                                                    fold_auc = roc_auc_score(fold_y_true, fold_y_proba)
                                                    fold_aucs.append(fold_auc)
                                                except ValueError:
                                                    continue
                                    
                                    if len(fold_aucs) > 1:
                                        auc_std.append(np.std(fold_aucs) / np.sqrt(len(fold_aucs)))
                                    else:
                                        auc_std.append(0)
                                        
                                except ValueError as e:
                                    print(f"    Warning: AUC calculation failed for {group_name} at time {t}: {e}")
                                    auc_values.append(np.nan)
                                    auc_std.append(0)
                            else:
                                # Only one class present
                                print(f"    Warning: Only class {unique_classes[0]} present for {group_name} at time {t} (n={len(time_trials)})")
                                auc_values.append(np.nan)
                                auc_std.append(0)
                        else:
                            # Not enough trials
                            auc_values.append(np.nan)
                            auc_std.append(0)
                    
                    # Match lengths with proper time axes
                    if time_axes is not None and len(time_axes) != len(auc_values):
                        if len(time_axes) > len(auc_values):
                            time_axes_group = time_axes[:len(auc_values)]
                        else:
                            time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(auc_values))
                    elif time_axes is not None:
                        time_axes_group = time_axes
                    else:
                        time_axes_group = np.arange(len(auc_values))
                    
                    # Plot AUC (only non-NaN values)
                    valid_indices = ~np.isnan(auc_values)
                    if np.any(valid_indices):
                        ax.plot(time_axes_group[valid_indices], np.array(auc_values)[valid_indices], 
                                   color=color, linestyle='-', linewidth=3, 
                                   label=f'{area}', alpha=0.8)
                        
                        # Plot error bars only for valid values
                        valid_auc = np.array(auc_values)[valid_indices]
                        valid_std = np.array(auc_std)[valid_indices]
                        valid_time = time_axes_group[valid_indices]
                        
                        ax.fill_between(valid_time, 
                                           valid_auc - valid_std,
                                           valid_auc + valid_std,
                                           alpha=0.1, color=color)
                    else:
                        print(f"    Warning: No valid AUC values for {group_name}")
                    
                    return time_axes_group
                return None
            
            # Process each heading group (each row)
            for i, (group_name, trials) in enumerate(all_trials_by_group.items()):
                # Plot Probability (left panel: columns 0, 1, 2)
                ax_prob = axes[i, j]
                time_axes_group = process_and_plot_probabilities(ax_prob, trials, group_name)
                
                # Plot AUC (right panel: columns 3, 4, 5)
                ax_auc = axes[i, j + 3]
                process_and_plot_auc(ax_auc, trials, group_name)
        
        # Format all subplots for this alignment (column j for prob, j+3 for AUC)
        for i, group_name in enumerate(current_heading_groups.keys()):
            # Format Probability subplot
            ax_prob = axes[i, j]
            ax_prob.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
            ax_prob.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            
            # Add velocity markers for stimOn alignment with text annotations
            if alignment == 'stimOn':
                ax_prob.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
                ax_prob.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
                
                # Add text annotations at the top of the plot
                ax_prob.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='orange', fontsize=10, transform=ax_prob.get_xaxis_transform())
                ax_prob.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='purple', fontsize=10, transform=ax_prob.get_xaxis_transform())
            
            # Add text annotation for alignment line with proper alignment name
            ax_prob.text(0, 0.95, alignment_names[alignment], rotation=90, verticalalignment='top', 
                       horizontalalignment='right', color='gray', fontsize=10,
                       transform=ax_prob.get_xaxis_transform())
            
            ax_prob.set_xlabel('Time (s)')
            ax_prob.set_ylabel(ylabel)
            
            # Add title for first column only
            if j == 0:
                ax_prob.set_title(f'{group_name}', fontsize=12, loc='left')
            
            # Add column title for first row only
            if i == 0:
                ax_prob.set_title(f'{alignment_names[alignment]}', fontsize=14)
            
            # Only show legend on the last probability subplot (bottom right of prob panels)
            if i == len(current_heading_groups) - 1 and j == len(alignments) - 1:
                ax_prob.legend(fontsize=8)
            
            ax_prob.set_ylim(0.0, 1.0)
            
            # Format AUC subplot
            ax_auc = axes[i, j + 3]
            ax_auc.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
            ax_auc.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
            
            # Add velocity markers for stimOn alignment with text annotations
            if alignment == 'stimOn':
                ax_auc.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
                ax_auc.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
                
                # Add text annotations at the top of the plot
                ax_auc.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='orange', fontsize=10, transform=ax_auc.get_xaxis_transform())
                ax_auc.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='purple', fontsize=10, transform=ax_auc.get_xaxis_transform())
            
            # Add text annotation for alignment line with proper alignment name
            ax_auc.text(0, 0.95, alignment_names[alignment], rotation=90, verticalalignment='top', 
                       horizontalalignment='right', color='gray', fontsize=10,
                       transform=ax_auc.get_xaxis_transform())
            
            ax_auc.set_xlabel('Time (s)')
            ax_auc.set_ylabel('AUC')
            
            # Add titles for AUC panels
            if i == 0:  # First row only
                ax_auc.set_title(f'{alignment_names[alignment]} - AUC', fontsize=14)
            
            # Only show legend on the last AUC subplot (bottom right of AUC panels)
            if i == len(current_heading_groups) - 1 and j == len(alignments) - 1:
                ax_auc.legend(fontsize=8)
            
            ax_auc.set_ylim(0.4, 1.0)
    
    # Add overall title
    fig.suptitle(f'{target.upper()} Decoding by Heading Difficulty\nLeft: Probabilities, Right: AUC (Recalculated for each heading group)', 
                 fontsize=16, y=0.98)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.95, left=0.05)
    
    # Save the figure
    save_filename = f"D:\\Neural-Pipeline\\results\\analysis_population\\compare_area_{target}_zarya_probability_all_headings_with_corrected_auc.png"
    plt.savefig(save_filename, dpi=300, bbox_inches='tight')
    print(f"\nFigure saved as: {save_filename}")
    
    plt.show()

### plot oneconf trial, prob and AUC

In [ ]:
target = 'PDW'  # Only PDW
print(f"\n{'#'*60}")
print(f"PROCESSING TARGET: {target.upper()} - BY oneTargConf AND PDW")
print(f"{'#'*60}")

# Function to create file path based on modality/coherence condition
def get_filepath(date, area, alignment, target, mod, coh):
    return f"D:\\Neural-Pipeline\\results\\analysis_population\\decoders\\zarya_{date}_{area}_{target}_{alignment}_train_mod{mod}_coh{coh}_test_mod{mod}_coh{coh}_cv_results.npy"

# Create figure with subplots (2 rows x 6 columns: 3 for probability, 3 for AUC)
fig, axes = plt.subplots(2, 6, figsize=(30, 12))
all_stats = {}

for j, alignment in enumerate(alignments):
    print(f"\n{'='*20} {alignment.upper()} {'='*20}")
    
    alignment_stats = {}
    
    # Get time axes for this alignment
    time_axes = time_axes_dots3DMP[alignment]
    
    for area in areas:
        color = area_colors[area]
        
        # Collect ALL individual trials from all dates, separated by oneTargConf and PDW
        all_trials_conf0_pdw0 = []    # oneTargConf == 0 & PDW == 0
        all_trials_conf0_pdw1 = []    # oneTargConf == 0 & PDW == 1
        all_trials_conf1_pdw0 = []    # oneTargConf == 1 & PDW == 0
        all_trials_conf1_pdw1 = []    # oneTargConf == 1 & PDW == 1
        all_trials_conf0_auc = []     # For AUC calculation
        all_trials_conf1_auc = []     # For AUC calculation
        
        print(f"\n{area}:")
        successful_dates = 0
        
        for mod_label, mod_cond in modality_conditions.items():
            mod = mod_cond['mod']
            coh = mod_cond['coh']
            
            for date in dates:
                # Load session data for this specific date
                try:
                    # Get time axes for this alignment (from loaded session data)
                    if time_axes is None:
                        time_axes = time_axes_dots3DMP[alignment]
                    
                    print(f"  {date}: Loaded session data successfully")
                except Exception as e:
                    print(f"  {date}: Failed to load session data - {str(e)}")
                    continue

                filepath = get_filepath(date, area, alignment, target, mod, coh)

                try:
                    # Load decoder results
                    results = np.load(filepath, allow_pickle=True).item()
                    trial_results = results['trial_results']

                    # Filter trials by oneTargConf and PDW using the stored behavioral data
                    for trial in trial_results:
                        if 'test_behavior' in trial:
                            conf_types = trial['test_behavior']['oneTargConf']

                            # Create separate trial entries for each test trial
                            for i, conf_type in enumerate(conf_types):
                                # Get PDW value for this trial
                                PDW = trial['test_behavior']['PDW'][i]
                                
                                # Create a single-trial entry
                                single_trial = {
                                    'time': trial['time'],
                                    'cv_fold': trial['cv_fold'],
                                    'y_proba': trial['y_proba'][i:i+1],  # Single prediction
                                    'y_pred': trial['y_pred'][i:i+1],
                                    'y_test': trial['y_test'][i:i+1],
                                    'auc': trial['auc'],  # Keep overall AUC (but won't use for filtered conditions)
                                    'accuracy': trial['accuracy'],  # Keep overall accuracy
                                    # Store all behavioral info for this specific trial
                                    'oneTargConf': conf_type,
                                    'heading': trial['test_behavior']['headingInd'][i],
                                    'choice': trial['test_behavior']['choice'][i],
                                    'PDW': PDW,
                                    'modality': trial['test_behavior']['modality'][i],
                                    'coherenceInd': trial['test_behavior']['coherenceInd'][i],
                                    'correct': trial['test_behavior']['correct'][i],
                                    'RT': trial['test_behavior']['RT'][i],
                                }
                                
                                # Sort into four categories based on oneTargConf and PDW
                                if conf_type == 0 and PDW == 0:  # oneTargConf=0, PDW=0
                                    all_trials_conf0_pdw0.append(single_trial)
                                elif conf_type == 0 and PDW == 1:  # oneTargConf=0, PDW=1
                                    all_trials_conf0_pdw1.append(single_trial)
                                elif conf_type == 1 and PDW == 0:  # oneTargConf=1, PDW=0
                                    all_trials_conf1_pdw0.append(single_trial)
                                elif conf_type == 1 and PDW == 1:  # oneTargConf=1, PDW=1
                                    all_trials_conf1_pdw1.append(single_trial)
                        
                        else:
                            # OLD FORMAT: Skip or handle differently
                            print(f"  Warning: Old format detected for {date}, skipping...")
                            continue
                    
                    successful_dates += 1
                    conf0_pdw0_count = len([t for t in trial_results if 'test_behavior' in t 
                                    for i, (c, p) in enumerate(zip(t['test_behavior']['oneTargConf'], t['test_behavior']['PDW'])) 
                                    if c == 0 and p == 0])
                    conf0_pdw1_count = len([t for t in trial_results if 'test_behavior' in t 
                                     for i, (c, p) in enumerate(zip(t['test_behavior']['oneTargConf'], t['test_behavior']['PDW'])) 
                                     if c == 0 and p == 1])
                    conf1_pdw0_count = len([t for t in trial_results if 'test_behavior' in t 
                                      for i, (c, p) in enumerate(zip(t['test_behavior']['oneTargConf'], t['test_behavior']['PDW'])) 
                                      if c == 1 and p == 0])
                    conf1_pdw1_count = len([t for t in trial_results if 'test_behavior' in t 
                                       for i, (c, p) in enumerate(zip(t['test_behavior']['oneTargConf'], t['test_behavior']['PDW'])) 
                                       if c == 1 and p == 1])
                    print(f"  {date}: Conf0PDW0: {conf0_pdw0_count}, Conf0PDW1: {conf0_pdw1_count}, Conf1PDW0: {conf1_pdw0_count}, Conf1PDW1: {conf1_pdw1_count}")
                    
                except FileNotFoundError:
                    print(f"  {date}: File not found - {filepath}")
                    continue
                except Exception as e:
                    print(f"  {date}: Error - {str(e)}")
                    continue
        
        if successful_dates == 0:
            print(f"  No data found for {area}")
            continue
        
        print(f"  Total - Conf0PDW0: {len(all_trials_conf0_pdw0)}, Conf0PDW1: {len(all_trials_conf0_pdw1)}")
        print(f"  Total - Conf1PDW0: {len(all_trials_conf1_pdw0)}, Conf1PDW1: {len(all_trials_conf1_pdw1)}")
        
        # Define function to process trials and plot probabilities for all conditions
        def process_and_plot_probabilities(ax, row_name):
            # Collect all trial groups for this row
            if row_name == "oneTargConf=0":
                trial_groups = [
                    (all_trials_conf0_pdw0, "Conf0PDW0", ":", 2),  # dotted, thin
                    (all_trials_conf0_pdw1, "Conf0PDW1", "-", 3)   # solid, thick
                ]
            else:  # row_name == "oneTargConf=1"
                trial_groups = [
                    (all_trials_conf1_pdw0, "Conf1PDW0", ":", 2),  # dotted, thin
                    (all_trials_conf1_pdw1, "Conf1PDW1", "-", 3)   # solid, thick
                ]
            
            for trials, condition_name, linestyle_base, linewidth_base in trial_groups:
                if trials:
                    time_points = sorted(list(set([trial['time'] for trial in trials])))
                    
                    class_0_proba = []
                    class_0_std = []
                    class_1_proba = []
                    class_1_std = []
                    
                    for t in time_points:
                        time_trials = [trial for trial in trials if trial['time'] == t]
                        
                        if time_trials:
                            class_0_probabilities = []
                            class_1_probabilities = []
                            
                            for trial in time_trials:
                                y_test = trial['y_test'][0]  # Single value now
                                y_proba = trial['y_proba'][0]  # Single value now
                                
                                if y_test == 0:
                                    class_0_probabilities.append(y_proba)
                                elif y_test == 1:
                                    class_1_probabilities.append(y_proba)
                            
                            # Calculate statistics
                            if class_0_probabilities:
                                class_0_proba.append(np.mean(class_0_probabilities))
                                class_0_std.append(np.std(class_0_probabilities) / np.sqrt(len(class_0_probabilities)))
                            else:
                                class_0_proba.append(np.nan)
                                class_0_std.append(0)
                            
                            if class_1_probabilities:
                                class_1_proba.append(np.mean(class_1_probabilities))
                                class_1_std.append(np.std(class_1_probabilities) / np.sqrt(len(class_1_probabilities)))
                            else:
                                class_1_proba.append(np.nan)
                                class_1_std.append(0)
                        else:
                            class_0_proba.append(np.nan)
                            class_0_std.append(0)
                            class_1_proba.append(np.nan)
                            class_1_std.append(0)
                    
                    # Match lengths with proper time axes
                    if len(time_axes) != len(class_0_proba):
                        if len(time_axes) > len(class_0_proba):
                            time_axes_plot = time_axes[:len(class_0_proba)]
                        else:
                            time_axes_plot = np.linspace(time_axes[0], time_axes[-1], len(class_0_proba))
                    else:
                        time_axes_plot = time_axes
                    
                    # Plot class 0 probabilities (dashed line)
                    class_0_label = f'{area} Low PDW ({condition_name})'
                    ax.plot(time_axes_plot, class_0_proba, color=color, linestyle='--', 
                               linewidth=linewidth_base-1, label=class_0_label, alpha=0.8)
                    ax.fill_between(time_axes_plot, 
                                       np.array(class_0_proba) - np.array(class_0_std),
                                       np.array(class_0_proba) + np.array(class_0_std),
                                       alpha=0.05, color=color)
                    
                    # Plot class 1 probabilities (solid line)
                    class_1_label = f'{area} High PDW ({condition_name})'
                    ax.plot(time_axes_plot, class_1_proba, color=color, linestyle=linestyle_base, 
                               linewidth=linewidth_base, label=class_1_label, alpha=0.8)
                    ax.fill_between(time_axes_plot, 
                                       np.array(class_1_proba) - np.array(class_1_std),
                                       np.array(class_1_proba) + np.array(class_1_std),
                                       alpha=0.05, color=color)

        
        # Define function to process trials and plot AUC for all conditions
        def process_and_plot_auc(ax, row_name):
            # Collect all trial groups for this row
            if row_name == "oneTargConf=0":
                # Combine all oneTargConf=0 trials (both PDW=0 and PDW=1)
                all_trials_conf0 = all_trials_conf0_pdw0 + all_trials_conf0_pdw1
                trial_groups = [
                    (all_trials_conf0, "oneTargConf=0", "-", 3)   # solid, thick
                ]
            else:  # row_name == "oneTargConf=1"
                # Combine all oneTargConf=1 trials (both PDW=0 and PDW=1)
                all_trials_conf1 = all_trials_conf1_pdw0 + all_trials_conf1_pdw1
                trial_groups = [
                    (all_trials_conf1, "oneTargConf=1", "-", 3)   # solid, thick
                ]
            
            for trials, condition_name, linestyle, linewidth in trial_groups:
                if trials:
                    time_points = sorted(list(set([trial['time'] for trial in trials])))
                    
                    auc_values = []
                    auc_std = []
                    
                    for t in time_points:
                        time_trials = [trial for trial in trials if trial['time'] == t]
                        
                        if len(time_trials) > 1:  # Need multiple trials to calculate AUC
                            # Collect y_true and y_proba for this time point and condition
                            y_true = []
                            y_proba = []
                            
                            for trial in time_trials:
                                y_true.append(trial['y_test'][0])  # This is the actual PDW (0 or 1)
                                y_proba.append(trial['y_proba'][0])  # This is the predicted probability of PDW=1
                            
                            # Convert to numpy arrays
                            y_true = np.array(y_true)
                            y_proba = np.array(y_proba)
                            
                            # Check if we have both classes (need both PDW=0 and PDW=1 for AUC)
                            unique_classes = np.unique(y_true)
                            if len(unique_classes) > 1:
                                try:
                                    # Calculate AUC for this specific condition and time point
                                    # This measures how well the decoder distinguishes PDW=0 vs PDW=1
                                    # within this oneTargConf condition
                                    auc = roc_auc_score(y_true, y_proba)
                                    auc_values.append(auc)
                                    
                                    # Calculate std across CV folds if available
                                    cv_folds = [trial['cv_fold'] for trial in time_trials]
                                    unique_folds = list(set(cv_folds))
                                    
                                    if len(unique_folds) > 1:
                                        # Calculate AUC for each CV fold separately
                                        fold_aucs = []
                                        for fold in unique_folds:
                                            fold_trials = [trial for trial in time_trials if trial['cv_fold'] == fold]
                                            if len(fold_trials) > 1:
                                                fold_y_true = [trial['y_test'][0] for trial in fold_trials]
                                                fold_y_proba = [trial['y_proba'][0] for trial in fold_trials]
                                                
                                                if len(np.unique(fold_y_true)) > 1:
                                                    fold_auc = roc_auc_score(fold_y_true, fold_y_proba)
                                                    fold_aucs.append(fold_auc)
                                        
                                        if len(fold_aucs) > 1:
                                            auc_std.append(np.std(fold_aucs) / np.sqrt(len(fold_aucs)))
                                        else:
                                            auc_std.append(0)
                                    else:
                                        auc_std.append(0)
                                        
                                except ValueError as e:
                                    # AUC calculation failed
                                    print(f"    Warning: AUC calculation failed for {condition_name} at time {t}: {e}")
                                    auc_values.append(np.nan)
                                    auc_std.append(0)
                            else:
                                # Only one class present, AUC is undefined
                                print(f"    Warning: Only one class present for {condition_name} at time {t} (class: {unique_classes[0]})")
                                auc_values.append(np.nan)
                                auc_std.append(0)
                        else:
                            # Not enough trials
                            auc_values.append(np.nan)
                            auc_std.append(0)
                    
                    # Match lengths with proper time axes
                    if len(time_axes) != len(auc_values):
                        if len(time_axes) > len(auc_values):
                            time_axes_plot = time_axes[:len(auc_values)]
                        else:
                            time_axes_plot = np.linspace(time_axes[0], time_axes[-1], len(auc_values))
                    else:
                        time_axes_plot = time_axes
                    
                    auc_label = f'{area} AUC ({condition_name})'
                    
                    # Plot AUC (only non-NaN values)
                    valid_indices = ~np.isnan(auc_values)
                    if np.any(valid_indices):
                        ax.plot(time_axes_plot[valid_indices], np.array(auc_values)[valid_indices], 
                                color=color, linestyle=linestyle, linewidth=linewidth, 
                                label=auc_label, alpha=0.8)
                        
                        # Plot error bars only for valid values
                        valid_auc = np.array(auc_values)[valid_indices]
                        valid_std = np.array(auc_std)[valid_indices]
                        valid_time = time_axes_plot[valid_indices]
                        
                        ax.fill_between(valid_time, 
                                        valid_auc - valid_std,
                                        valid_auc + valid_std,
                                        alpha=0.05, color=color)
                    else:
                        print(f"    Warning: No valid AUC values for {condition_name}")
                        
        # Plot probabilities for both rows
        process_and_plot_probabilities(axes[0, j], "oneTargConf=0")  # Row 0: oneTargConf=0
        process_and_plot_probabilities(axes[1, j], "oneTargConf=1")  # Row 1: oneTargConf=1
        
        # Plot AUC for both rows
        process_and_plot_auc(axes[0, j+3], "oneTargConf=0")  # Row 0: oneTargConf=0
        process_and_plot_auc(axes[1, j+3], "oneTargConf=1")  # Row 1: oneTargConf=1
    
    # Format all subplots for this alignment
    row_titles = ["oneTargConf=0", "oneTargConf=1"]
    
    for row in range(2):
        # Format probability plots (left 3 columns)
        ax_prob = axes[row, j]
        ax_prob.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
        ax_prob.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        
        # Add velocity markers for stimOn alignment with text annotations
        if alignment == 'stimOn':
            ax_prob.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
            ax_prob.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
            
            # Add text annotations at the top of the plot
            ax_prob.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                       rotation=90, verticalalignment='top', horizontalalignment='right',
                       color='orange', fontsize=10, transform=ax_prob.get_xaxis_transform())
            ax_prob.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                       rotation=90, verticalalignment='top', horizontalalignment='right',
                       color='purple', fontsize=10, transform=ax_prob.get_xaxis_transform())
        
        # Add text annotation for alignment line with proper alignment name
        ax_prob.text(0, 0.95, alignment_names[alignment], rotation=90, verticalalignment='top', 
                   horizontalalignment='right', color='gray', fontsize=10,
                   transform=ax_prob.get_xaxis_transform())
        
        ax_prob.set_xlabel('Time (s)')
        ax_prob.set_ylabel('Probability of High PDW')
        ax_prob.set_title(f'{alignment_names[alignment]} - {row_titles[row]} (Prob)', fontsize=12)
        
        # Only show legend on the last subplot of each row
        if j == len(alignments) - 1:
            ax_prob.legend(fontsize=8)
        
        ax_prob.set_ylim(0.0, 1.0)
        
        # Format AUC plots (right 3 columns)
        ax_auc = axes[row, j+3]
        ax_auc.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
        ax_auc.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        
        # Add velocity markers for stimOn alignment with text annotations
        if alignment == 'stimOn':
            ax_auc.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
            ax_auc.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
            
            # Add text annotations at the top of the plot
            ax_auc.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                       rotation=90, verticalalignment='top', horizontalalignment='right',
                       color='orange', fontsize=10, transform=ax_auc.get_xaxis_transform())
            ax_auc.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                       rotation=90, verticalalignment='top', horizontalalignment='right',
                       color='purple', fontsize=10, transform=ax_auc.get_xaxis_transform())
        
        # Add text annotation for alignment line with proper alignment name
        ax_auc.text(0, 0.95, alignment_names[alignment], rotation=90, verticalalignment='top', 
                   horizontalalignment='right', color='gray', fontsize=10,
                   transform=ax_auc.get_xaxis_transform())
        
        ax_auc.set_xlabel('Time (s)')
        ax_auc.set_ylabel('AUC')
        ax_auc.set_title(f'{alignment_names[alignment]} - {row_titles[row]} (AUC)', fontsize=12)
        
        # Only show legend on the last subplot of each row
        if j == len(alignments) - 1:
            ax_auc.legend(fontsize=8)
        
        ax_auc.set_ylim(0.0, 1.0)

# Add overall title
fig.suptitle(f'PDW Decoding by oneTargConf and PDW\nLeft: Probabilities, Right: AUC (Recalculated for filtered conditions)\nUpper: oneTargConf=0, Lower: oneTargConf=1', 
             fontsize=16, y=0.98)

plt.tight_layout()
plt.subplots_adjust(top=0.90)

# Save the figure
save_filename = f"D:\\Neural-Pipeline\\results\\analysis_population\\compare_area_PDW_zarya_probability_and_auc_by_oneTargConf_and_PDW.png"
plt.savefig(save_filename, dpi=300, bbox_inches='tight')
print(f"\nFigure saved as: {save_filename}")

plt.show()

### plot correct / error

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

# Function to create file path
def get_filepath(date, area, alignment, target):
    return f"D:\\Neural-Pipeline\\results\\analysis_population\\decoders\\zarya_{date}_{area}_{target}_{alignment}_train_mod1_coh1_test_mod1_coh1_cv_results.npy"

# Focus on stimulus only
target = 'choice'  # Only choice
print(f"\n{'#'*60}")
print(f"PROCESSING TARGET: {target.upper()} - CORRECT vs ERROR TRIALS BY HEADING")
print(f"{'#'*60}")

# Define specific heading groups for error analysis (±1.5° and ±3.9°)
valid_heading_groups = {
    'Combined_Headings': [1, 2, 3, 5, 6, 7]  # ±1.5° (indices 1,6) and ±3.9° (indices 2,7)
}

print("Using combined heading groups: ±1.5° and ±3.9° headings")

# First pass: collect all data
all_data = {}

for j, alignment in enumerate(alignments):
    print(f"\n{'='*20} {alignment.upper()} {'='*20}")
    
    # Initialize time_axes
    time_axes = None
    all_data[alignment] = {}
    
    for area in areas:
        print(f"\n{area}:")
        
        # Collect trials for the combined heading group, separated by correct/error
        all_trials_high = []
        all_trials_low = []
        
        successful_dates = 0
        
        for date in dates:
            # Load session data for this specific date
            try:
                time_axes_dots3DMP = data_dict['time_axes_dots3DMP']
                
                # Get time axes for this alignment (from loaded session data)
                if time_axes is None:
                    time_axes = time_axes_dots3DMP[alignment]
                
                print(f"  {date}: Loaded session data successfully")
            except Exception as e:
                print(f"  {date}: Failed to load session data - {str(e)}")
                continue
            
            filepath = get_filepath(date, area, alignment, target)
            
            try:
                # Load decoder results
                results = np.load(filepath, allow_pickle=True).item()
                trial_results = results['trial_results']
                
                # Filter trials by heading difficulty and correctness
                for trial in trial_results:
                    # Check if trial has the new format with behavioral data
                    if 'test_behavior' in trial:
                        # NEW FORMAT: Extract heading info directly from trial results
                        headings = trial['test_behavior']['headingInd']
                        trial_flags = trial['test_behavior']['correct']
                        
                        # Create separate trial entries for each test trial
                        for i, heading in enumerate(headings):
                            # Only include trials with headings in our target group
                            if heading in valid_heading_groups['Combined_Headings']:
                                # Create a single-trial entry
                                single_trial = {
                                    'time': trial['time'],
                                    'cv_fold': trial['cv_fold'],
                                    'y_proba': trial['y_proba'][i:i+1],  # Single prediction
                                    'y_pred': trial['y_pred'][i:i+1],
                                    'y_test': trial['y_test'][i:i+1],
                                    'auc': trial['auc'],  # Keep overall AUC (but won't use for filtered conditions)
                                    'accuracy': trial['accuracy'],  # Keep overall accuracy
                                    # Store all behavioral info for this specific trial
                                    'heading': heading,
                                    'choice': trial['test_behavior']['choice'][i],
                                    'PDW': trial['test_behavior']['PDW'][i],
                                    'modality': trial['test_behavior']['modality'][i],
                                    'coherenceInd': trial['test_behavior']['coherenceInd'][i],
                                    'correct': trial['test_behavior']['correct'][i],
                                    'oneTargConf': trial['test_behavior']['oneTargConf'][i],
                                    'RT': trial['test_behavior']['RT'][i],
                                }
                                
                                # Add trial to appropriate group based on correctness
                                if trial_flags[i] == 1:  # Correct trial
                                    all_trials_high.append(single_trial)
                                else:  # Error trial
                                    all_trials_low.append(single_trial)
                    
                    else:
                        # OLD FORMAT: Skip or handle differently
                        print(f"  Warning: Old format detected for {date}, skipping...")
                        continue
                
                successful_dates += 1
                
            except FileNotFoundError:
                print(f"  {date}: File not found - {filepath}")
                continue
            except Exception as e:
                print(f"  {date}: Error - {str(e)}")
                continue
        
        if successful_dates == 0:
            print(f"  No data found for {area}")
            continue
        
        # Print total trial counts
        correct_count = len(all_trials_high)
        error_count = len(all_trials_low)
        print(f"Correct: {correct_count}, Error: {error_count}")
        
        # Store data for this area and alignment
        all_data[alignment][area] = {
            'correct': all_trials_high,
            'error': all_trials_low,
            'time_axes': time_axes
        }

# Create figure with subplots (3 rows x 6 columns: areas x (3 prob + 3 AUC))
n_rows = len(areas)  # One row per brain area
fig, axes = plt.subplots(n_rows, 6, figsize=(30, 5*n_rows))

# Handle single row case
if n_rows == 1:
    axes = axes.reshape(1, -1)

# Set ylabel
ylabel = 'Probability of Right Heading'

# Now plot each brain area as a separate row
for i, area in enumerate(areas):
    color = area_colors[area]
    
    for j, alignment in enumerate(alignments):
        correct_trials = all_data[alignment][area]['correct']
        error_trials = all_data[alignment][area]['error']
        
        # DEBUG: Print trial counts and class distribution
        print(f"\n=== DEBUGGING {area}-{alignment} ===")
        print(f"Correct trials: {len(correct_trials)}, Error trials: {len(error_trials)}")
        
        if error_trials:
            error_class_0_count = sum(1 for trial in error_trials if trial['y_test'][0] == 0)
            error_class_1_count = sum(1 for trial in error_trials if trial['y_test'][0] == 1)
            print(f"Error trials - Class 0 (Left): {error_class_0_count}, Class 1 (Right): {error_class_1_count}")
        
        if correct_trials:
            correct_class_0_count = sum(1 for trial in correct_trials if trial['y_test'][0] == 0)
            correct_class_1_count = sum(1 for trial in correct_trials if trial['y_test'][0] == 1)
            print(f"Correct trials - Class 0 (Left): {correct_class_0_count}, Class 1 (Right): {correct_class_1_count}")
        
        # ===== PLOT PROBABILITIES =====
        ax_prob = axes[i, j]
        time_axes_group = None
        
        # Process correct trials
        if correct_trials:
            time_points = sorted(list(set([trial['time'] for trial in correct_trials])))
            
            all_dv = []
            class_0_std = []
            class_1_proba = []
            class_1_std = []
            
            for t in time_points:
                time_trials = [trial for trial in correct_trials if trial['time'] == t]
                
                if time_trials:
                    class_0_probabilities = []
                    class_1_probabilities = []
                    
                    for trial in time_trials:
                        y_test = trial['y_test'][0]  # Single value now
                        y_proba = trial['y_proba'][0]  # Single value now
                        
                        if y_test == 0:
                            class_0_probabilities.append(y_proba)
                        elif y_test == 1:
                            class_1_probabilities.append(y_proba)
                    
                    # Calculate statistics
                    if class_0_probabilities:
                        all_dv.append(np.mean(class_0_probabilities))
                        class_0_std.append(np.std(class_0_probabilities) / np.sqrt(len(class_0_probabilities)))
                    else:
                        all_dv.append(np.nan)
                        class_0_std.append(0)
                    
                    if class_1_probabilities:
                        class_1_proba.append(np.mean(class_1_probabilities))
                        class_1_std.append(np.std(class_1_probabilities) / np.sqrt(len(class_1_probabilities)))
                    else:
                        class_1_proba.append(np.nan)
                        class_1_std.append(0)
                else:
                    all_dv.append(np.nan)
                    class_0_std.append(0)
                    class_1_proba.append(np.nan)
                    class_1_std.append(0)
            
            # Get time axes from stored data
            time_axes = all_data[alignment][area]['time_axes']
            
            # Match lengths with proper time axes
            if time_axes is not None and len(time_axes) != len(all_dv):
                if len(time_axes) > len(all_dv):
                    time_axes_group = time_axes[:len(all_dv)]
                else:
                    time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(all_dv))
            elif time_axes is not None:
                time_axes_group = time_axes
            else:
                time_axes_group = np.arange(len(all_dv))
            
            # Plot correct trials with solid lines (only valid values)
            valid_dv = ~np.isnan(all_dv)
            valid_class_1 = ~np.isnan(class_1_proba)
            
            if np.any(valid_dv):
                ax_prob.plot(time_axes_group[valid_dv], np.array(all_dv)[valid_dv], 
                           color="red", linestyle='-', linewidth=2, label=f'Left Heading (Correct)', alpha=0.8)
                ax_prob.fill_between(time_axes_group[valid_dv], 
                                   (np.array(all_dv) - np.array(class_0_std))[valid_dv],
                                   (np.array(all_dv) + np.array(class_0_std))[valid_dv],
                                   alpha=0.1, color="red")
            
            if np.any(valid_class_1):
                ax_prob.plot(time_axes_group[valid_class_1], np.array(class_1_proba)[valid_class_1], 
                           color="blue", linestyle='-', linewidth=3, label=f'Right Heading (Correct)', alpha=0.8)
                ax_prob.fill_between(time_axes_group[valid_class_1], 
                                   (np.array(class_1_proba) - np.array(class_1_std))[valid_class_1],
                                   (np.array(class_1_proba) + np.array(class_1_std))[valid_class_1],
                                   alpha=0.1, color="blue")
        
        # Process error trials
        if error_trials:
            time_points = sorted(list(set([trial['time'] for trial in error_trials])))
            
            class_0_proba_err = []
            class_0_std_err = []
            class_1_proba_err = []
            class_1_std_err = []
            
            for t in time_points:
                time_trials = [trial for trial in error_trials if trial['time'] == t]
                
                if time_trials:
                    class_0_probabilities = []
                    class_1_probabilities = []
                    
                    for trial in time_trials:
                        y_test = trial['y_test'][0]  # Single value now
                        y_proba = trial['y_proba'][0]  # Single value now
                        
                        if y_test == 0:
                            class_0_probabilities.append(y_proba)
                        elif y_test == 1:
                            class_1_probabilities.append(y_proba)
                    
                    # Calculate statistics
                    if class_0_probabilities:
                        class_0_proba_err.append(np.mean(class_0_probabilities))
                        class_0_std_err.append(np.std(class_0_probabilities) / np.sqrt(len(class_0_probabilities)))
                    else:
                        class_0_proba_err.append(np.nan)
                        class_0_std_err.append(0)
                    
                    if class_1_probabilities:
                        class_1_proba_err.append(np.mean(class_1_probabilities))
                        class_1_std_err.append(np.std(class_1_probabilities) / np.sqrt(len(class_1_probabilities)))
                    else:
                        class_1_proba_err.append(np.nan)
                        class_1_std_err.append(0)
                else:
                    class_0_proba_err.append(np.nan)
                    class_0_std_err.append(0)
                    class_1_proba_err.append(np.nan)
                    class_1_std_err.append(0)
            
            # Get time axes from stored data
            time_axes = all_data[alignment][area]['time_axes']
            
            # Match lengths with proper time axes
            if time_axes is not None and len(time_axes) != len(class_0_proba_err):
                if len(time_axes) > len(class_0_proba_err):
                    time_axes_group = time_axes[:len(class_0_proba_err)]
                else:
                    time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(class_0_proba_err))
            elif time_axes is not None:
                time_axes_group = time_axes
            else:
                time_axes_group = np.arange(len(class_0_proba_err))
            
            # Plot error trials with dashed lines (only valid values)
            valid_class_0_err = ~np.isnan(class_0_proba_err)
            valid_class_1_err = ~np.isnan(class_1_proba_err)
            
            print(f"Error plotting - Valid class 0: {np.sum(valid_class_0_err)}, Valid class 1: {np.sum(valid_class_1_err)}")
            
            if np.any(valid_class_0_err):
                ax_prob.plot(time_axes_group[valid_class_0_err], np.array(class_0_proba_err)[valid_class_0_err], 
                           color="red", linestyle='--', linewidth=2, label=f'Left Heading (Error)', alpha=0.6)
                ax_prob.fill_between(time_axes_group[valid_class_0_err], 
                                   (np.array(class_0_proba_err) - np.array(class_0_std_err))[valid_class_0_err],
                                   (np.array(class_0_proba_err) + np.array(class_0_std_err))[valid_class_0_err],
                                   alpha=0.05, color="red")
            
            if np.any(valid_class_1_err):
                ax_prob.plot(time_axes_group[valid_class_1_err], np.array(class_1_proba_err)[valid_class_1_err], 
                           color="blue", linestyle='--', linewidth=2, label=f'Right Heading (Error)', alpha=0.6)
                ax_prob.fill_between(time_axes_group[valid_class_1_err], 
                                   (np.array(class_1_proba_err) - np.array(class_1_std_err))[valid_class_1_err],
                                   (np.array(class_1_proba_err) + np.array(class_1_std_err))[valid_class_1_err],
                                   alpha=0.05, color="blue")
        
        # ===== PLOT AUC =====
        ax_auc = axes[i, j + 3]
        time_axes_group = None
        
        # Process correct trials AUC
        if correct_trials:
            time_points = sorted(list(set([trial['time'] for trial in correct_trials])))
            
            auc_values = []
            auc_std = []
            
            for t in time_points:
                time_trials = [trial for trial in correct_trials if trial['time'] == t]
                
                if len(time_trials) >= 2:  # Need at least 2 trials
                    # Collect y_true and y_proba for this time point
                    y_true = []
                    y_proba = []
                    
                    for trial in time_trials:
                        y_true.append(trial['y_test'][0])
                        y_proba.append(trial['y_proba'][0])
                    
                    # Convert to numpy arrays
                    y_true = np.array(y_true)
                    y_proba = np.array(y_proba)
                    
                    # Check if we have both classes
                    unique_classes = np.unique(y_true)
                    if len(unique_classes) > 1:
                        try:
                            # Calculate AUC for this time point
                            overall_auc = roc_auc_score(y_true, y_proba)
                            auc_values.append(overall_auc)
                            
                            # Calculate std across CV folds if available
                            cv_folds = [trial['cv_fold'] for trial in time_trials]  # FIXED
                            unique_folds = list(set(cv_folds))
                            
                            fold_aucs = []
                            for fold in unique_folds:
                                fold_trials = [trial for trial in time_trials if trial['cv_fold'] == fold]
                                if len(fold_trials) >= 2:
                                    fold_y_true = np.array([trial['y_test'][0] for trial in fold_trials])
                                    fold_y_proba = np.array([trial['y_proba'][0] for trial in fold_trials])
                                    
                                    if len(np.unique(fold_y_true)) > 1:
                                        try:
                                            fold_auc = roc_auc_score(fold_y_true, fold_y_proba)
                                            fold_aucs.append(fold_auc)
                                        except ValueError:
                                            continue
                            
                            if len(fold_aucs) > 1:
                                auc_std.append(np.std(fold_aucs) / np.sqrt(len(fold_aucs)))
                            else:
                                auc_std.append(0)
                                
                        except ValueError as e:
                            auc_values.append(np.nan)
                            auc_std.append(0)
                    else:
                        auc_values.append(np.nan)
                        auc_std.append(0)
                else:
                    auc_values.append(np.nan)
                    auc_std.append(0)
            
            # Get time axes from stored data
            time_axes = all_data[alignment][area]['time_axes']
            
            # Match lengths with proper time axes
            if time_axes is not None and len(time_axes) != len(auc_values):
                if len(time_axes) > len(auc_values):
                    time_axes_group = time_axes[:len(auc_values)]
                else:
                    time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(auc_values))
            elif time_axes is not None:
                time_axes_group = time_axes
            else:
                time_axes_group = np.arange(len(auc_values))
            
            # Plot AUC for correct trials (solid line)
            valid_indices = ~np.isnan(auc_values)
            if np.any(valid_indices):
                ax_auc.plot(time_axes_group[valid_indices], np.array(auc_values)[valid_indices], 
                           color="black", linestyle='-', linewidth=3, 
                           label=f'Correct', alpha=0.8)
                
                # Plot error bars only for valid values
                valid_auc = np.array(auc_values)[valid_indices]
                valid_std = np.array(auc_std)[valid_indices]
                valid_time = time_axes_group[valid_indices]
                
                ax_auc.fill_between(valid_time, 
                                   valid_auc - valid_std,
                                   valid_auc + valid_std,
                                   alpha=0.1, color="black")
        
        # Process error trials AUC
        if error_trials:
            time_points = sorted(list(set([trial['time'] for trial in error_trials])))
            
            auc_values_err = []
            auc_std_err = []
            
            for t in time_points:
                time_trials = [trial for trial in error_trials if trial['time'] == t]  # FIXED
                
                if len(time_trials) >= 2:  # Need at least 2 trials
                    # Collect y_true and y_proba for this time point
                    y_true = []
                    y_proba = []
                    
                    for trial in time_trials:
                        y_true.append(trial['y_test'][0])
                        y_proba.append(trial['y_proba'][0])
                    
                    # Convert to numpy arrays
                    y_true = np.array(y_true)
                    y_proba = np.array(y_proba)
                    
                    # Check if we have both classes
                    unique_classes = np.unique(y_true)
                    if len(unique_classes) > 1:
                        try:
                            # Calculate AUC for this time point
                            overall_auc = roc_auc_score(y_true, y_proba)
                            auc_values_err.append(overall_auc)
                            
                            # Calculate std across CV folds if available
                            cv_folds = [trial['cv_fold'] for trial in time_trials]  # FIXED
                            unique_folds = list(set(cv_folds))
                            
                            fold_aucs = []
                            for fold in unique_folds:
                                fold_trials = [trial for trial in time_trials if trial['cv_fold'] == fold]
                                if len(fold_trials) >= 2:
                                    fold_y_true = np.array([trial['y_test'][0] for trial in fold_trials])
                                    fold_y_proba = np.array([trial['y_proba'][0] for trial in fold_trials])
                                    
                                    if len(np.unique(fold_y_true)) > 1:
                                        try:
                                            fold_auc = roc_auc_score(fold_y_true, fold_y_proba)
                                            fold_aucs.append(fold_auc)
                                        except ValueError:
                                            continue
                            
                            if len(fold_aucs) > 1:
                                auc_std_err.append(np.std(fold_aucs) / np.sqrt(len(fold_aucs)))
                            else:
                                auc_std_err.append(0)
                                
                        except ValueError as e:
                            auc_values_err.append(np.nan)
                            auc_std_err.append(0)
                    else:
                        auc_values_err.append(np.nan)
                        auc_std_err.append(0)
                else:
                    auc_values_err.append(np.nan)
                    auc_std_err.append(0)
            
            # Get time axes from stored data
            time_axes = all_data[alignment][area]['time_axes']
            
            # Match lengths with proper time axes
            if time_axes is not None and len(time_axes) != len(auc_values_err):
                if len(time_axes) > len(auc_values_err):
                    time_axes_group = time_axes[:len(auc_values_err)]
                else:
                    time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(auc_values_err))
            elif time_axes is not None:
                time_axes_group = time_axes
            else:
                time_axes_group = np.arange(len(auc_values_err))
            
            # Plot AUC for error trials (dashed line)
            valid_indices = ~np.isnan(auc_values_err)
            print(f"AUC Error plotting - Valid indices: {np.sum(valid_indices)}")
            
            if np.any(valid_indices):
                ax_auc.plot(time_axes_group[valid_indices], np.array(auc_values_err)[valid_indices], 
                           color="black", linestyle='--', linewidth=2, 
                           label=f'Error', alpha=0.6)
                
                # Plot error bars only for valid values
                valid_auc = np.array(auc_values_err)[valid_indices]
                valid_std = np.array(auc_std_err)[valid_indices]
                valid_time = time_axes_group[valid_indices]
                
                ax_auc.fill_between(valid_time, 
                                   valid_auc - valid_std,
                                   valid_auc + valid_std,
                                   alpha=0.05, color="black")

# Format all subplots
for j, alignment in enumerate(alignments):
    for i, area in enumerate(areas):
        # Format Probability subplot
        ax_prob = axes[i, j]
        ax_prob.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
        ax_prob.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        
        # Add velocity markers for stimOn alignment with text annotations
        if alignment == 'stimOn':
            ax_prob.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
            ax_prob.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
            
            # Add text annotations at the top of the plot
            ax_prob.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                       rotation=90, verticalalignment='top', horizontalalignment='right',
                       color='orange', fontsize=10, transform=ax_prob.get_xaxis_transform())
            ax_prob.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                       rotation=90, verticalalignment='top', horizontalalignment='right',
                       color='purple', fontsize=10, transform=ax_prob.get_xaxis_transform())
        
        ax_prob.set_ylim(0, 1)
        ax_prob.set_ylabel(ylabel if j == 0 else '')
        ax_prob.set_title(f'{area} - {alignment}' if i == 0 else '')
        ax_prob.legend(fontsize=8)
        ax_prob.grid(True, alpha=0.3)
        
        # Format AUC subplot
        ax_auc = axes[i, j + 3]
        ax_auc.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
        ax_auc.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        
        # Add velocity markers for stimOn alignment
        if alignment == 'stimOn':
            ax_auc.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
            ax_auc.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
        
        ax_auc.set_ylim(0, 1)
        ax_auc.set_ylabel('AUC' if j == 0 else '')
        ax_auc.set_title(f'{area} - {alignment} AUC' if i == 0 else '')
        ax_auc.legend(fontsize=8)
        ax_auc.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### plot high / low confidence

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import os

# Decision variable calculation function
def calculate_decision_variable(y_test, y_proba, epsilon=1e-10):
    """
    Calculate decision variable from classifier probabilities
    
    Args:
        y_test: True label (0 or 1)
        y_proba: Classifier probability for class 1
        epsilon: Small value to avoid log(0)
    
    Returns:
        dv: Decision variable (positive = confident, negative = uncertain)
    """
    # Ensure y_proba is in valid range [epsilon, 1-epsilon]
    y_proba_clipped = np.clip(y_proba, epsilon, 1 - epsilon)
    
    # Calculate log-odds (decision variable)
    dv = np.log(y_proba_clipped / (1 - y_proba_clipped))
    
    # If true class is 0, flip the sign
    if y_test == 0:
        dv = -dv
    
    return dv

# Create output directories
output_dir_correct = "D:\\Neural-Pipeline\\results\\analysis_population\\@prob_auc_sortbycorrect"
output_dir_pdw = "D:\\Neural-Pipeline\\results\\analysis_population\\@prob_auc_sortbypdw"
os.makedirs(output_dir_correct, exist_ok=True)
os.makedirs(output_dir_pdw, exist_ok=True)

# Function to create file path
def get_filepath(date, area, alignment, target, mod, coh):
    return f"D:\\Neural-Pipeline\\results\\analysis_population\\decoders\\zarya_{date}_{area}_{target}_{alignment}_train_mod{mod}_coh{coh}_test_mod{mod}_coh{coh}_cv_results.npy"

# Define targets and conditions to loop through
targets = ['stimulus', 'choice']

# Define specific heading groups for error analysis (±1.5° and ±3.9°)
valid_heading_groups = {
    'Combined_Headings': [1, 2, 3, 5, 6, 7]  # ±1.5° (indices 1,6) and ±3.9° (indices 2,7)
}

# Loop through each target
for target in targets:
    print(f"\n{'#'*80}")
    print(f"PROCESSING TARGET: {target.upper()} - HIGH vs LOW PDW TRIALS BY HEADING")
    print(f"{'#'*80}")
    
    # Loop through each modality/coherence condition
    for condition_name, condition_params in modality_conditions.items():
        mod = condition_params['mod']
        coh = condition_params['coh']
        
        print(f"\n{'='*60}")
        print(f"CONDITION: {condition_name} (mod={mod}, coh={coh})")
        print(f"{'='*60}")
        
        # First pass: collect all data
        all_data = {}

        for j, alignment in enumerate(alignments):
            print(f"\n{'='*20} {alignment.upper()} {'='*20}")
            
            # Initialize time_axes
            time_axes = None
            all_data[alignment] = {}
            
            for area in areas:
                print(f"\n{area}:")
                
                # Collect trials separated by PDW and correctness
                all_high_correct = []
                all_high_error = []
                all_low_correct = []
                all_low_error = []  
                
                successful_dates = 0
                
                for date in dates:
                    # Load session data for this specific date
                    try:
                        time_axes_dots3DMP = data_dict['time_axes_dots3DMP']
                        
                        # Get time axes for this alignment (from loaded session data)
                        if time_axes is None:
                            time_axes = time_axes_dots3DMP[alignment]
                        
                        print(f"  {date}: Loaded session data successfully")
                    except Exception as e:
                        print(f"  {date}: Failed to load session data - {str(e)}")
                        continue
                    
                    filepath = get_filepath(date, area, alignment, target, mod, coh)
                    
                    try:
                        # Load decoder results
                        results = np.load(filepath, allow_pickle=True).item()
                        trial_results = results['trial_results']
                        
                        # Filter trials by heading difficulty and PDW
                        for trial in trial_results:
                            # Check if trial has the new format with behavioral data
                            if 'test_behavior' in trial:
                                headings = trial['test_behavior']['headingInd']
                                pdw_flags = trial['test_behavior']['PDW'] 
                                correct_flags = trial['test_behavior']['correct']
                                
                                # Create separate trial entries for each test trial
                                for i, heading in enumerate(headings):
                                    # Only include trials with headings in our target group
                                    if heading in valid_heading_groups['Combined_Headings']:
                                        # Extract values for DV calculation
                                        y_test_val = trial['y_test'][i]
                                        y_proba_val = trial['y_proba'][i]
                                        
                                        # Calculate decision variable
                                        dv = calculate_decision_variable(y_test_val, y_proba_val)
                                        
                                        # Create a single-trial entry
                                        single_trial = {
                                            'time': trial['time'],
                                            'cv_fold': trial['cv_fold'],
                                            'y_proba': trial['y_proba'][i:i+1],  # Single prediction
                                            'y_pred': trial['y_pred'][i:i+1],
                                            'y_test': trial['y_test'][i:i+1],
                                            'dv': dv,  
                                            'auc': trial['auc'],
                                            'accuracy': trial['accuracy'],
                                            # Store all behavioral info for this specific trial
                                            'heading': heading,
                                            'choice': trial['test_behavior']['choice'][i],
                                            'PDW': trial['test_behavior']['PDW'][i],
                                            'modality': trial['test_behavior']['modality'][i],
                                            'coherenceInd': trial['test_behavior']['coherenceInd'][i],
                                            'correct': trial['test_behavior']['correct'][i],
                                            'oneTargConf': trial['test_behavior']['oneTargConf'][i],
                                            'RT': trial['test_behavior']['RT'][i],
                                        }
                                        
                                        # Add trial to appropriate group based on PDW and correctness
                                        if pdw_flags[i] == 1:  # High PDW trial
                                            if correct_flags[i] == 1:  # Correct trial
                                                all_high_correct.append(single_trial)
                                            else:  # Error trial
                                                all_high_error.append(single_trial)
                                        else:  # Low PDW trial
                                            if correct_flags[i] == 1:  # Correct trial
                                                all_low_correct.append(single_trial)
                                            else:  # Error trial
                                                all_low_error.append(single_trial)
                            
                            else:
                                # OLD FORMAT: Skip or handle differently
                                print(f"  Warning: Old format detected for {date}, skipping...")
                                continue
                        
                        successful_dates += 1
                        
                    except FileNotFoundError:
                        print(f"  {date}: File not found - {filepath}")
                        continue
                    except Exception as e:
                        print(f"  {date}: Error - {str(e)}")
                        continue
                
                if successful_dates == 0:
                    print(f"  No data found for {area}")
                    continue
                
                print(f"  High PDW Correct: {len(all_high_correct)}, High PDW Error: {len(all_high_error)}")
                print(f"  Low PDW Correct: {len(all_low_correct)}, Low PDW Error: {len(all_low_error)}")    
                
                # Store data with correct keys
                all_data[alignment][area] = {
                    'high_correct': all_high_correct,  
                    'high_error': all_high_error,
                    'low_correct': all_low_correct,  
                    'low_error': all_low_error,
                    'time_axes': time_axes
                }

                hc_count = len(all_data[alignment][area]['high_correct'])
                he_count = len(all_data[alignment][area]['high_error'])
                lc_count = len(all_data[alignment][area]['low_correct'])
                le_count = len(all_data[alignment][area]['low_error'])
                
                # Find minimum 
                min_trials = min(hc_count, he_count, lc_count, le_count)
                

        # Create figure with subplots (3 rows x 3 columns: areas x alignments)
        n_rows = len(areas)  # One row per brain area
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5*n_rows))

        # Handle single row case
        if n_rows == 1:
            axes = axes.reshape(1, -1)

        # Set ylabel
        ylabel = 'Decision Variable (DV), {target}'

        # Function to process and plot DV for a group of trials
        def plot_dv_group(trials, ax, time_axes, min_trials, color, linestyle, label, alpha=0.8):
            if not trials:
                return
            if len(trials) > min_trials:
                np.random.seed(42)  # For reproducibility
                indices = np.random.choice(len(trials), min_trials, replace=False)
                trials = [trials[i] for i in indices]

            time_points = sorted(list(set([trial['time'] for trial in trials])))
            
            dv_mean = []
            dv_std = []
            
            for t in time_points:
                time_trials = [trial for trial in trials if trial['time'] == t]
                
                if time_trials:
                    all_dv = [trial['dv'] for trial in time_trials]
                    dv_mean.append(np.mean(all_dv))
                    dv_std.append(np.std(all_dv) / np.sqrt(len(all_dv)))
                else:
                    dv_mean.append(np.nan)
                    dv_std.append(0)
            
            # Match lengths with proper time axes
            if time_axes is not None and len(time_axes) != len(dv_mean):
                if len(time_axes) > len(dv_mean):
                    time_axes_group = time_axes[:len(dv_mean)]
                else:
                    time_axes_group = np.linspace(time_axes[0], time_axes[-1], len(dv_mean))
            elif time_axes is not None:
                time_axes_group = time_axes
            else:
                time_axes_group = np.arange(len(dv_mean))
            
            # Plot only valid values
            valid_dv = ~np.isnan(dv_mean)
            
            if np.any(valid_dv):
                ax.plot(time_axes_group[valid_dv], np.array(dv_mean)[valid_dv], 
                       color=color, linestyle=linestyle, linewidth=2, label=label, alpha=alpha)
                ax.fill_between(time_axes_group[valid_dv], 
                               (np.array(dv_mean) - np.array(dv_std))[valid_dv],
                               (np.array(dv_mean) + np.array(dv_std))[valid_dv],
                               alpha=0.1, color=color)

        # Now plot each brain area as a separate row
        for i, area in enumerate(areas):
            color = area_colors[area]
            
            for j, alignment in enumerate(alignments):
                # Get the data
                high_correct_trials = all_data[alignment][area]['high_correct']
                high_error_trials = all_data[alignment][area]['high_error']
                low_correct_trials = all_data[alignment][area]['low_correct']       
                low_error_trials = all_data[alignment][area]['low_error']
                
                # Get time axes
                time_axes = all_data[alignment][area]['time_axes']
                
                # Plot DV
                ax = axes[i, j]
                
                # Plot all four groups
                plot_dv_group(high_correct_trials, ax, time_axes, min_trials, 'blue', '-', 'High PDW Correct', 0.8)
                plot_dv_group(high_error_trials, ax, time_axes, min_trials, 'red', '-', 'High PDW Error', 0.3)
                plot_dv_group(low_correct_trials, ax, time_axes, min_trials, 'blue', '--', 'Low PDW Correct', 0.8)
                plot_dv_group(low_error_trials, ax, time_axes,min_trials, 'red', '--', 'Low PDW Error', 0.3)

        # Format all subplots
        for j, alignment in enumerate(alignments):
            for i, area in enumerate(areas):
                # Format DV subplot
                ax = axes[i, j]
                ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
                ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
                
                # Add velocity markers for stimOn alignment with text annotations
                if alignment == 'stimOn':
                    ax.axvline(x=vel_markers['max_acceleration'], color='orange', linestyle=':', alpha=0.7)
                    ax.axvline(x=vel_markers['max_velocity'], color='purple', linestyle=':', alpha=0.7)
                    
                    # Add text annotations at the top of the plot
                    ax.text(vel_markers['max_acceleration'], 0.95, 'Max Acc', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='orange', fontsize=10, transform=ax.get_xaxis_transform())
                    ax.text(vel_markers['max_velocity'], 0.95, 'Max Vel', 
                           rotation=90, verticalalignment='top', horizontalalignment='right',
                           color='purple', fontsize=10, transform=ax.get_xaxis_transform())
                
                ax.set_ylabel(ylabel if j == 0 else '')
                ax.set_title(f'{area} - {alignment}' if i == 0 else '')
                ax.legend(fontsize=8)
                ax.grid(True, alpha=0.3)

        # Add overall title for the figure
        fig.suptitle(f'{target.upper()} Decoder - {condition_name} (mod={mod}, coh={coh}) - Decision Variable by PDW & Correctness', 
                     fontsize=16, y=0.98)
        
        plt.tight_layout()
        
        # Save the figure
        filename = f"{target}_{condition_name}_mod{mod}_coh{coh}_dv_pdw_correct.png"
        filepath = os.path.join(output_dir_pdw, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"\nSaved figure: {filepath}")
        
        plt.show()
        
        # Save the data as well
        data_filename = f"{target}_{condition_name}_mod{mod}_coh{coh}_dv_pdw_correct_data.npy"
        data_filepath = os.path.join(output_dir_pdw, data_filename)
        np.save(data_filepath, all_data, allow_pickle=True)
        print(f"Saved data: {data_filepath}")

print(f"\n{'='*80}")
print("DV ANALYSIS COMPLETE!")
print(f"Results saved to:")
print(f"  - {output_dir_pdw}")
print(f"{'='*80}")